# Experiment 3: Replicate with DeepSeek

## Setup

In [1]:
from nest_asyncio import apply
from os import environ
from llama_index.llms.replicate import Replicate

apply()
# environ['REPLICATE_API_TOKEN'] = ''

llm_replicate = \
    Replicate(
        model='deepseek-ai/deepseek-r1', 
        api_token=environ['REPLICATE_API_TOKEN'],
    )

## Load documents

In [20]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, SimpleDirectoryReader

embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5')

Settings.llm = llm_replicate
Settings.embed_model = embed_model

overview_statement_pdf = SimpleDirectoryReader(input_files=['../data/overview_statement.pdf']).load_data()
task_breakdown_pdf = SimpleDirectoryReader(input_files=['../data/task_breakdown.pdf']).load_data()

## Interpret documents

In [19]:
from llama_index.core import VectorStoreIndex

overview_statement = \
    VectorStoreIndex \
        .from_documents(overview_statement_pdf) \
        .as_query_engine(similarity_top_k=3) \
        .query('Tell me about the project.')

task_breakdown = \
    VectorStoreIndex \
        .from_documents(task_breakdown_pdf) \
        .as_query_engine(similarity_top_k=3) \
        .query('List all project tasks.')

print(overview_statement, task_breakdown)

<think>
Okay, let's see. I need to answer the query "Tell me about the project" using the provided context. Let me start by reading through the context information carefully. 

The context is from a PDF about Chicago WideCast Smart-Home Services. The project overview mentions that they want to automate all their business workflows using generative AI, specifically a conversational AI assistant for customers and employees. 

First, I should outline the key components of the project. The company offers various services like online TV plans, data plans, movie streaming, PPV, video games, home security, and utilities. Each service has different tiers or options.

Then there are four roles: Managers, Account Specialists, Technical Support Specialists, and Customers. Each role has specific permissions and workflows. For example, Managers can add/update/delete products and orders, while Customers can manage their accounts, pay bills, and handle orders.

The business rules include renting the 

## Start chat

In [21]:
from llama_index.core.llms import ChatMessage

def any_message(role, sections):
    message = f"Ask the {role} to predict the amount of effort required to complete sections:\n\n"
    for _, section in enumerate(sections):
        message += f'- {section}\n'
    return message

print(
    llm_replicate.chat(
        [
            ChatMessage(
                role='system', 
                content= \
                    'Consider the overview statement:\n\n. ' + \
                    f"{overview_statement}:\n\n." + \
                    'The tasks for the software project are:\n\n' + \
                    f"{task_breakdown}:\n\n." + \
                    'Consider the overview statement document of the project Chicago WideCase Smart-Home Services. ' + \
                    'Given the work listed in the task description document\n\n.' + \
                    '- Tag each task with a unique ID that starts with `REQ-001`.\n' + \
                    '- Create effort estimate for each task.',
            ),
            ChatMessage(
                role='user', 
                content= \
                    any_message(
                        'Project Manager',
                        ['Project plan', 'Risk Mitigation and Contingency Plan'],
                    ),
            ),
            ChatMessage(
                role='user', 
                content= \
                    any_message(
                        'Requirement Engineer',
                        ['Requirement'],
                    ),
            ),
            ChatMessage(
                role='user', 
                content= \
                    any_message(
                        'System Engineer',
                        ['Analysis', 'Design'],
                    ),
            ),
            ChatMessage(
                role='user', 
                content= \
                    any_message(
                        'Test Engineer',
                        ['Testing'],
                    ),
            ),
            ChatMessage(
                role='user', 
                content= \
                    any_message(
                        'Documentation Engineer',
                        ['Documentation'],
                    ),
            ),
            ChatMessage(
                role='user', 
                content= 'Summarize the effort estimation.',
            ),
        ],
    ),
)

assistant: <think>
Alright, let me go through this step by step. The user wants me to assign unique IDs starting with REQ-001 to each task, create effort estimates, and summarize the effort estimation based on different roles. Let me start by understanding the existing tasks.

First, the tasks are organized under different sections like Project Plan, Risk Mitigation, etc. Each section has its own set of tasks, some with sub-tasks. The ID needs to start with REQ-001, so maybe each main task gets a unique ID, and sub-tasks can have sub-IDs like REQ-001.1, REQ-001.2, etc. But the exact format isn't specified, so I'll need to assign a unique ID to each task and sub-task, sequentially.

Next, effort estimation. The context mentions work amounts in hours and productivity rates. For example, under Project Plan, Write Plan has a work amount of 40 hours and productivity rate of 1.0, giving 40 hours. Similarly, others have similar data. Each task's effort is calculated by Work Amount * Productiv